In [4]:

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from sklearn.metrics.pairwise import cosine_similarity


# 1. Load the embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Our document chunks
chunks = [
    "Python is a programming language.",
    "Machine learning enables computers to learn from data.",
    "Kubernetes manages containerized applications."
]

# 3. Convert chunks into embeddings
chunk_vectors = embedding_model.encode(chunks).tolist()

print("Number of chunk vectors:", len(chunk_vectors))
print("Vector dimensions:", len(chunk_vectors[0]))

# 4. Create an in-memory Qdrant database
client = QdrantClient(path="./qdrant_data")

# 5. Create a collection with HNSW indexing
client.create_collection(
    collection_name="rag_demo",
    vectors_config=models.VectorParams(
        size=len(chunk_vectors[0]),
        distance=models.Distance.COSINE
    ),
    hnsw_config=models.HnswConfigDiff(
        m=16,
        ef_construct=100,
        full_scan_threshold=0
    )
)

# 6. Store vectors, chunk IDs and original text
client.upsert(
    collection_name="rag_demo",
    points=[
        models.PointStruct(
            id=i,
            vector=chunk_vectors[i],
            payload={"text": chunks[i]}
        )
        for i in range(len(chunks))
    ],
    wait=True
)

print("Chunks stored successfully!")

# 7. User submits a query
query = "Which tool manages containers?"

# 8. Convert the query into an embedding
query_vector = embedding_model.encode(query).tolist()

# 9. Search the HNSW index
results = client.query_points(
    collection_name="rag_demo",
    query=query_vector,
    limit=2,
    with_payload=True
).points

# 10. Display the matching chunks
print("\nSearch results:")

for result in results:
    print("Chunk ID:", result.id)
    print("Similarity score:", round(result.score, 4))
    print("Chunk text:", result.payload["text"])
    print("-" * 40)
# Compare query embedding with every chunk embedding
scores = cosine_similarity(
    [query_vector],
    chunk_vectors
)[0]

print("\nDirect cosine similarity scores:")

for i, score in enumerate(scores):
    print(f"Chunk {i}: {score:.4f} | {chunks[i]}")


Number of chunk vectors: 3
Vector dimensions: 384
Chunks stored successfully!

Search results:
Chunk ID: 2
Similarity score: 0.6049
Chunk text: Kubernetes manages containerized applications.
----------------------------------------
Chunk ID: 0
Similarity score: 0.1055
Chunk text: Python is a programming language.
----------------------------------------

Direct cosine similarity scores:
Chunk 0: 0.1055 | Python is a programming language.
Chunk 1: 0.0969 | Machine learning enables computers to learn from data.
Chunk 2: 0.6049 | Kubernetes manages containerized applications.
